In [4]:
# !pip install pykrx
!pip install pandas --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 88.4 MB/s eta 0:00:00:00:01:01
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pykrx 1.2.4 requires pandas<3.0,>=2.2.0, but you have pandas 3.0.2 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have nump

In [1]:
import pprint

class DataDict(dict):
    """
    데이터 저장 Dictionary
    built-in: dict의 확장으로 저장 요소에 대해 attribute 접근 방식을 허용
    기본 제공 Alias (별칭): dD, dDict

    사용 예시)
        myData = DataDict(name='JEHYEUK', age=34, division='Vehicle Solution Team')
        print(myData.name, myData['name'], myData.name == myData['name'])

        /* ----------------------------------------------------------------------------------------
        | 결과
        -------------------------------------------------------------------------------------------
        | JEHYEUK JEHYEUK True
        ---------------------------------------------------------------------------------------- */
    """
    def __init__(self, data=None, **kwargs):
        super().__init__()

        data = data or {}
        data.update(kwargs)
        for key, value in data.items():
            if isinstance(value, dict):
                value = DataDict(**value)
            self[key] = value

    def __getattr__(self, attr):
        if attr in self:
            return self[attr]
        return super().__getattribute__(attr)

    def __setattr__(self, attr, value):
        if isinstance(value, dict):
            self[attr] = DataDict(**value)
        else:
            self[attr] = value

    def __str__(self) -> str:
        return pprint.pformat(self)


# Alias
DataDictionary = DataDict

In [2]:
from datetime import datetime, timedelta
from pytz import timezone
from pykrx.stock import get_nearest_business_day_in_a_week
from requests.exceptions import ConnectionError, HTTPError, SSLError
from typing import Union


class TradingDate:

    tz = timezone('Asia/Seoul')
    def __init__(self):
        return

    def __str__(self) -> str:
        return self.latest

    def __sub__(self, other:int):
        key = f"{self.closed}m{other}"
        if not hasattr(self, key):
            prev = datetime.strptime(self.closed, "%Y%m%d") - timedelta(other)
            setattr(self, key, get_nearest_business_day_in_a_week(prev.strftime("%Y%m%d")))
        return getattr(self, key)

    @classmethod
    def clock(cls, fmt:str="") -> Union[datetime, str]:
        if (fmt == "") or (fmt is None):
            return datetime.now(cls.tz)
        return datetime.now(cls.tz).strftime(fmt)


    def get_closed(self) -> str:
        if self.is_open():
            base = (self.clock().date() - timedelta(days=1)).strftime("%Y%m%d")
        else:
            base = self.clock("%Y%m%d")

        try:
            return get_nearest_business_day_in_a_week(base)
        except (ConnectionError, HTTPError, SSLError, IndexError, Exception):
            return base


    @property
    def closed(self) -> str:
        if not hasattr(self, '_closed'):
            setattr(self, '_closed', self.get_closed())
        return getattr(self, '_closed')

    @closed.setter
    def closed(self, closed:str):
        setattr(self, '_closed', closed)

    @property
    def latest(self) -> str:
        if not hasattr(self, '_latest'):
            try:
                setattr(self, '_latest', get_nearest_business_day_in_a_week())
            except (ConnectionError, HTTPError, SSLError, IndexError, Exception):
                setattr(self, '_latest', self.clock('%Y%m%d'))
        return getattr(self, '_latest')

    def is_open(self) -> bool:
        return (self.clock("%Y%m%d") == self.latest) and (900 <= int(self.clock("%H%M")) <= 1530)

In [3]:
from datetime import datetime, timedelta
from pandas import concat, Series, DataFrame
from pykrx.stock import get_market_ohlcv_by_date, get_market_cap_by_date


def get_ohlcv(**kwargs) -> DataFrame:
    ohlcv = get_market_ohlcv_by_date(**kwargs)
    trade_stop = ohlcv[ohlcv.시가 == 0].copy()
    if not trade_stop.empty:
        ohlcv.loc[trade_stop.index, ['시가', '고가', '저가']] = trade_stop.종가
    ohlcv.index.name = 'date'
    return ohlcv.rename(columns=dict(시가='open', 고가='high', 저가='low', 종가='close', 거래량='volume'))[[
        'open', 'high', 'low', 'close', 'volume'
    ]]

def get_market_cap(**kwargs) -> DataFrame:
    return get_market_cap_by_date(**kwargs)

class krx:
    get_ohlcv = staticmethod(get_ohlcv)
    get_market_cap = staticmethod(get_market_cap)

In [5]:
if not 'DataDict' in globals():
    from pylabwons.schema.datadict import DataDict

URLS = lambda ticker: DataDict(
    SNAPSHOT=f"http://comp.fnguide.com/SVO2/ASP/SVD_Main.asp?" \
             f"pGB=1&" \
             f"gicode=A{ticker}&" \
             f"cID=&" \
             f"MenuYn=Y" \
             f"&ReportGB=" \
             f"&NewMenuID=" \
             f"&stkGb=701",
    BANDS=f"http://cdn.fnguide.com/SVO2/json/chart/01_06/chart_A{ticker}_D.json",
    FOREIGNRATE3M=f"http://cdn.fnguide.com/SVO2/json/chart/01_01/chart_A{ticker}_3M.json",
    FOREIGNRATE1Y=f"http://cdn.fnguide.com/SVO2/json/chart/01_01/chart_A{ticker}_1Y.json",
    FOREIGNRATE3Y=f"http://cdn.fnguide.com/SVO2/json/chart/01_01/chart_A{ticker}_3Y.json",
    PRODUCTS=f"http://cdn.fnguide.com/SVO2/json/chart/02/chart_A{ticker}_01_N.json",
    EXPENSES=f"http://cdn.fnguide.com/SVO2/json/chart/02/chart_A{ticker}_D.json",
    XML=f"http://cdn.fnguide.com/SVO2/xml/Snapshot_all/{ticker}.xml"
)

HEADER = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/133.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "ko-KR,ko;q=0.9",
    "Referer": "https://comp.fnguide.com/",
    "Connection": "keep-alive",
}

LABEL_ESTIMATION = {
    "투자의견": "estimation",
    "목표주가": "targetPrice",
    "EPS": "estimatedEps",
    "PER": "estimatedPe",
    "추정기관수": "nOfEstimations"
}

KEY_CHANGE_RATE = {
    "영업이익": "profit",
    "당기순이익": "netProfit",
    "자산총계": "asset",
    "부채총계": "debt",
    "영업이익률": "profitRate",
    "EPS": "eps",
    "DPS": "dps"
}

# IFRS 공시가 없는 종목
NUMBER_EXCEPTION = ['088980', '094800', '415640']

SCHEMA = DataDict(
    URLS=URLS,
    HEADER=HEADER,
    LABEL_ESTIMATION=LABEL_ESTIMATION,
    KEY_CHANGE_RATE=KEY_CHANGE_RATE,
    NUMBER_EXCEPTION=NUMBER_EXCEPTION
)

In [6]:
if not "SCHEMA" in globals():
    from pylabwons.schema import fnguide as SCHEMA
from functools import cached_property
from io import StringIO
from lxml import html
from pandas import DataFrame, Series
from typing import List, Union
import numpy as np
import pandas as pd
import random, requests, time


class FnGuide:

    def __init__(self, ticker: str):
        self.URL = SCHEMA.URLS(ticker)
        self.ticker = ticker
        return

    @staticmethod
    def _fetch(url: str, encoding: str = "utf-8") -> requests.Response:
        session = requests.Session()

        for attempt in range(5):
            try:
                resp = session.get(url, headers=SCHEMA.HEADER, timeout=(3, 10))
                if resp.status_code == 200:
                    resp.encoding = encoding
                    return resp

                if resp.status_code in (502, 503, 504):
                    time.sleep(3 + random.random() * 5)
                    continue

                if resp.status_code in (403, 429):
                    raise PermissionError(f"Blocked: {resp.status_code}")

            except requests.RequestException:
                time.sleep(3 + random.random() * 5)
        raise ConnectionError(f"Failed to fetch after retries: {url}")
    
    @staticmethod
    def _separate_confirmed_estimated(st: DataFrame):
        """
        잠정 실적(P)이 존재하는 경우 잠정 실적을 확정 실적에 준하여 계산에 사용
        """
        _confirmed = st[~st.index.str.contains(r"\(|\)")].copy()
        _estimated = st[~st.index.isin(_confirmed.index)].copy()
        if _estimated.index[0].endswith('(P)'):
            prov = _estimated.iloc[[0]].copy()  # 최근 잠정 실적
            _confirmed = pd.concat([_confirmed, prov], axis=0)
            _estimated.drop(index=_estimated.index[0], inplace=True)
        return _confirmed, _estimated

    @staticmethod
    def _typecast(value: str) -> Union[int, float, str]:
        if str(value) in ['', ' ', '-', 'nan', '완전잠식', "N/A(IFRS)"]:
            return np.nan

        value = str(value) \
            .replace(" ", "") \
            .replace("%", "") \
            .replace(",", "")
        if any([c in value for c in ['/', '*']]) or all([c.isalpha() for c in value]):
            return value
        value = value.lower()
        if not any([char.isdigit() for char in value]):
            return np.nan
        return float(value) if "." in value or "-" in value else int(value)

    def _src2statement(self, src: DataFrame) -> DataFrame:
        data = src.set_index(keys=[src.columns[0]])
        if isinstance(data.columns[0], tuple):
            data.columns = data.columns.droplevel()
        else:
            data.columns = data.iloc[0]
            data = data.drop(index=data.index[0])
        data = data.T
        data.index.name = '기말'
        data.index = [
            idx.replace("(E) : Estimate 컨센서스, 추정치 ", "").replace("(P) : Provisional 잠정실적 ", "")
            for idx in data.index
        ]
        data.columns.name = None
        data = data.map(self._typecast) \
            .drop(columns=[col for col in data.columns if "발표기준" in col]) \
            .rename(columns={col: col[:col.find("(")] if "(" in col else col for col in data.columns})
        return data
    
    def _statement2numbers(self, yy: DataFrame, qq: DataFrame) -> Series:

        def __growth(arr: np.ndarray):
            if abs(arr[0]) <= 0.1:
                return np.nan
            if arr[0] < 0 < arr[-1]:
                return 9999.9999  # 흑자 전환
            if arr[0] > 0 > arr[-1]:
                return -9999.9999  # 적자 전환
            if arr[0] < 0 and arr[-1] < 0:
                return -9999.9998  # 적자 지속
            return round(100 * (arr[-1] - arr[0]) / abs(arr[0]), 2)

        def __payout_ratio(arr: np.ndarray):
            shares, dps, net_profit = arr
            if net_profit <= 0:
                return np.nan
            return round(100 * (1000 * shares) * dps / (net_profit * 1e+8), 2)

        # 확정 실적과 추정/잠정 실적 분리
        confirmed_yy, estimated_yy = self._separate_confirmed_estimated(yy)
        confirmed_qq, _ = self._separate_confirmed_estimated(qq)
        trailing = confirmed_qq.iloc[-4:].sum()

        # 매출처 이름
        sales_col = confirmed_yy.columns[0]
        fiscal_month = confirmed_yy.index[-1]

        # 영업이익률 계산 일원화
        confirmed_yy['영업이익률'] = round(100 * confirmed_yy['영업이익'] / confirmed_yy[sales_col], 2)
        confirmed_qq['영업이익률'] = round(100 * confirmed_qq['영업이익'] / confirmed_qq[sales_col], 2)
        estimated_yy['영업이익률'] = round(100 * estimated_yy['영업이익'] / estimated_yy[sales_col], 2)

        # 계산용 데이터
        # - 정적 데이터: 확정/잠정 실적 + 추정 실적 1개년
        # - 동적 데이터: 확정/잠정 실적 + 추정 실적 1개년에 대한 변화율
        #   @[{"매출", "이자수익", "보험수익"}, "영업이익", "당기순이익",
        #     "자산총계", "부채총계", "영업이익률", "EPS", "DPS"]
        columns = SCHEMA.KEY_CHANGE_RATE.copy()
        columns.update({sales_col: "revenue", "배당성향": "payoutRatio"})

        static = pd.concat([confirmed_yy, estimated_yy.iloc[[0]]])

        # 배당성향 계산
        # - 잠정 실적에 DPS가 존재하는 경우, 직전 확정 실적과 최근 4분기 합산 DPS 중 큰 값으로 대체
        if fiscal_month.endswith('(P)') and not pd.isna(static.at[fiscal_month, 'DPS']):
            static.at[fiscal_month, 'DPS'] \
                = max(static.at[confirmed_yy.index[-2], 'DPS'], trailing['DPS'])
        static['발행주식수'] = static['발행주식수'].ffill()
        static['배당성향'] = static[['발행주식수', 'DPS', '당기순이익']].apply(__payout_ratio, axis=1, raw=True)

        # 변화율(성장률) 계산
        # - 잠정 실적이 존재하는 경우, 결측치는 직전 확정실적으로 대체
        dynamic_fiscal = static[columns.keys()] \
            .rename(columns=columns) \
            .rolling(2) \
            .apply(__growth, raw=True) \
            .replace({9999.9999: "흑자전환", -9999.9999: "적자전환", -9999.9998: "적자지속"})
        if fiscal_month.endswith('(P)'):
            rebase = static.ffill().copy()
        else:
            rebase = static.copy()
        dynamic_estimated = rebase[columns.keys()] \
            .rename(columns=columns) \
            .rolling(2) \
            .apply(__growth, raw=True) \
            .replace({9999.9999: "흑자전환", -9999.9999: "적자전환", -9999.9998: "적자지속"})
        yoy = confirmed_qq[[c for c in columns if c in confirmed_qq.columns]] \
            .rename(columns=columns) \
            .rolling(5) \
            .apply(__growth, raw=True) \
            .replace({9999.9999: "흑자전환", -9999.9999: "적자전환", -9999.9998: "적자지속"}) \
            .iloc[-1][['revenue', 'profit', 'netProfit', 'profitRate', 'eps']]
        yoy.index = [f'yoy{col[0].upper()}{col[1:]}' for col in yoy.index]

        # 최근 4분기 합산 실적 및 최근 결산년도 주요 확정 실적 취합
        # - 최근 결산년도 EPS는 별도 계산
        # - 결측치는 직전 회계년도 실적으로 대체
        fiscal = static.loc[fiscal_month].copy()
        fiscal = fiscal.combine_first(static.loc[confirmed_yy.index[-2]])
        data = Series(data=dict(
            trailingRevenue=trailing[sales_col],
            trailingProfit=trailing['영업이익'],
            trailingNetProfit=trailing['당기순이익'],
            trailingProfitRate=100 * trailing['영업이익'] / trailing[sales_col] if trailing[sales_col] > 0 else np.nan,
            trailingEps=trailing['EPS'],
            fiscalMonth=fiscal.name,
            fiscalRevenue=fiscal[sales_col],
            fiscalProfit=fiscal['영업이익'],
            fiscalNetProfit=fiscal['당기순이익'],
            fiscalAsset=fiscal['자산총계'],
            fiscalCapital=fiscal['자본총계'],
            fiscalDebt=fiscal['부채총계'],
            fiscalDebtRatio=fiscal['부채비율'],
            fiscalRetentionRate=fiscal['유보율'],
            fiscalProfitRate=100 * fiscal['영업이익'] / fiscal[sales_col] if fiscal[sales_col] > 0 else np.nan,
            fiscalDps=fiscal['DPS'],
            # fiscalEps=fiscal['EPS'], # 직전 EPS 별도 계산 (중복 방지)
            fiscalDividendYield=fiscal['배당수익률'],
            fiscalPayoutRatio=fiscal['배당성향'],
            returnOnAsset=fiscal['ROA'],
            returnOnEquity=fiscal['ROE'],
        ))
        fiscal_pct = dynamic_fiscal.loc[fiscal_month]
        fiscal_pct = fiscal_pct.combine_first(dynamic_fiscal.loc[confirmed_yy.index[-2]])
        for col in columns.values():
            data[f'fiscal{col[0].upper() + col[1:]}Growth'] = fiscal_pct[col]
        for i, val in yoy.items():
            data[i] = val

        # 최근 추정년도 기준 실적
        data['estimatedMonth'] = est = estimated_yy.index[0]
        estimated = static.loc[est]
        for key, col in columns.items():
            if col == 'eps':
                continue  # @estimation과 중복 방지
            data[f'estimated{col[0].upper() + col[1:]}'] = estimated[key]

        estimated_pct = dynamic_estimated.loc[est]
        for col in columns.values():
            data[f'estimated{col[0].upper() + col[1:]}Growth'] = estimated_pct[col]

        return data

    @cached_property
    def _snapshot_text(self) -> str:
        return self._fetch(self.URL.SNAPSHOT).text

    @cached_property
    def _snapshot_tables(self) -> List[DataFrame]:
        return pd.read_html(StringIO(self._snapshot_text), header=0, displayed_only=False)

    @property
    def annual_statement(self) -> DataFrame:
        return self.annual_statement_consolidate if self.gb == 'D' else self.annual_statement_separate

    @cached_property
    def annual_statement_consolidate(self) -> DataFrame:
        if len(self._snapshot_tables) == 17:
            n = 11
        elif len(self._snapshot_tables) == 15:
            n = 9
        else:
            raise IndexError(f"Unexpected number of snapshot tables for {self.ticker}")
        return self._src2statement(self._snapshot_tables[n])
    
    @cached_property
    def annual_statement_confirmed(self) -> DataFrame:
        confirmed, _ = self._separate_confirmed_estimated(self.annual_statement)
        return confirmed
    
    @cached_property
    def annual_statement_estimated(self) -> DataFrame:
        _, estimated = self._separate_confirmed_estimated(self.annual_statement)
        return estimated

    @cached_property
    def annual_statement_separate(self) -> DataFrame:
        if len(self._snapshot_tables) == 17:
            n = 14
        elif len(self._snapshot_tables) == 15:
            n = 12
        else:
            raise IndexError(f"Unexpected number of snapshot tables for {self.ticker}")
        return self._src2statement(self._snapshot_tables[n])

    @cached_property
    def date(self) -> str:
        return html.fromstring(self._snapshot_text).xpath('//span[@class="date"]//text()')[0][1:-1].replace("/", "")

    @cached_property
    def estimation(self) -> Series:
        if len(self._snapshot_tables) == 17:
            n = 7
        elif len(self._snapshot_tables) == 15:
            n = 5
        else:
            raise IndexError(f"Unexpected number of snapshot tables for {self.ticker}")
        data = self._snapshot_tables[n].rename(columns=SCHEMA.LABEL_ESTIMATION).T[0]
        return data.map(self._typecast)

    @cached_property
    def gb(self) -> str:
        sy = self.annual_statement_separate
        cy = self.annual_statement_consolidate
        sq = self.quarter_statement_separate
        cq = self.quarter_statement_consolidate
        if (sy.count().sum() > cy.count().sum()) or (sq.count().sum() > cq.count().sum()):
            return 'B'  # 별도
        else:
            return 'D'  # 연결

    @property
    def quarter_statement(self) -> DataFrame:
        return self.quarter_statement_consolidate if self.gb == 'D' else self.quarter_statement_separate

    @cached_property
    def quarter_statement_consolidate(self) -> DataFrame:
        if len(self._snapshot_tables) == 17:
            n = 12
        elif len(self._snapshot_tables) == 15:
            n = 10
        else:
            raise IndexError(f"Unexpected number of snapshot tables for {self.ticker}")
        return self._src2statement(self._snapshot_tables[n])
    
    @cached_property
    def quarter_statement_confirmed(self) -> DataFrame:
        confirmed, _ = self._separate_confirmed_estimated(self.quarter_statement)
        return confirmed
    
    @cached_property
    def quarter_statement_estimated(self) -> DataFrame:
        _, estimated = self._separate_confirmed_estimated(self.quarter_statement)
        return estimated

    @cached_property
    def quarter_statement_separate(self) -> DataFrame:
        if len(self._snapshot_tables) == 17:
            n = 15
        elif len(self._snapshot_tables) == 15:
            n = 13
        else:
            raise IndexError(f"Unexpected number of snapshot tables for {self.ticker}")
        return self._src2statement(self._snapshot_tables[n])

    @cached_property
    def numbers(self) -> Series:
        table = self._snapshot_tables[0]
        fifty_two_weeks = table.iloc[0, 1].replace(' ', '').split("/")
        shares_outstanding = table.iloc[5, 1].replace(' ', '').split('/')
        float_shares = table.iloc[6, 1].replace(' ', '').split('/')

        data = Series(data=dict(
            numbersDate=self.date,
            close=table.columns[1].replace(' ', '').split("/")[0],
            fiftyTwoWeekHigh=fifty_two_weeks[0],
            fiftyTwoWeekLow=fifty_two_weeks[1],
            foreignRate=table.iloc[1, 3],
            beta=table.iloc[2, 3],
            sharesOutstanding=shares_outstanding[0],
            sharesPreferred=shares_outstanding[1],
            sharesFloating=float_shares[0],
            sharesFloatingRate=float_shares[1],
            ifrsType=self.gb if not self.ticker in SCHEMA.NUMBER_EXCEPTION else np.nan,
        ))

        tree = html.fromstring(self._snapshot_text)
        for dl in tree.xpath('//div[@id="corp_group2"]/dl'):
            key = "".join(dl.xpath('.//a[contains(@class, "tip_in")]//text()')).strip()
            if key == "PER":
                data['fiscalPe'] = dl.xpath('./dd/text()')[0].strip()
            if key == "12M PER":
                data['forwardPe'] = dl.xpath('./dd/text()')[0].strip()
            if key == "업종 PER":
                data['industryPe'] = dl.xpath('./dd/text()')[0].strip()
            if key == "PBR":
                data['fiscalPriceToBook'] = dl.xpath('./dd/text()')[0].strip()
        data = data.map(self._typecast)

        if data.fiscalPe > 0:
            data['fiscalEps'] = round(data.close / data.fiscalPe, 2)
        data['forwardEps'] = round(data.close / data.forwardPe, 2) if data.forwardPe > 0 else np.nan
        if not self.ticker in SCHEMA.NUMBER_EXCEPTION:
            data = pd.concat([
                data,
                self.estimation,
                self._statement2numbers(self.annual_statement, self.quarter_statement)
            ])
        data.name = self.ticker
        if not data.index.is_unique:
            data = data.sort_values(na_position='last') \
                .groupby(level=0) \
                .head(1)
        return data



# fng = FnGuide('005930')
# fng.annual_statement
# fng.quarter_statement
# fng.quarter_statement_confirmed
# fng.snapshot
# fng.per_band
# fng.foreign_rate
# fng.products
# fng.expenses


In [8]:
from pykrx.website.comm import webio
from requests import Session


def login_krx(login_id: str, login_pw: str, logger=print):
    """
    KRX data.krx.co.kr 로그인 후 세션 쿠키(JSESSIONID)를 갱신합니다.

    로그인 흐름:
    1. GET MDCCOMS001.cmd  → 초기 JSESSIONID 발급
    2. GET login.jsp       → iframe 세션 초기화
    3. POST MDCCOMS001D1.cmd → 실제 로그인
    4. CD011(중복 로그인) → skipDup=Y 추가 후 재전송
    """
    _LOGIN_PAGE = "https://data.krx.co.kr/contents/MDC/COMS/client/MDCCOMS001.cmd"
    _LOGIN_JSP  = "https://data.krx.co.kr/contents/MDC/COMS/client/view/login.jsp?site=mdc"
    _LOGIN_URL  = "https://data.krx.co.kr/contents/MDC/COMS/client/MDCCOMS001D1.cmd"
    _UA = (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"
    )

    # 초기 세션 발급
    _session = Session()
    _session.get(_LOGIN_PAGE, headers={"User-Agent": _UA}, timeout=15)
    _session.get(_LOGIN_JSP, headers={"User-Agent": _UA, "Referer": _LOGIN_PAGE}, timeout=15)

    payload = {
        "mbrNm": "", "telNo": "", "di": "", "certType": "",
        "mbrId": login_id, "pw": login_pw,
    }
    headers = {"User-Agent": _UA, "Referer": _LOGIN_PAGE}

    # 로그인 POST
    resp = _session.post(_LOGIN_URL, data=payload, headers=headers, timeout=15)
    data = resp.json()
    error_code = data.get("_error_code", "")

    # CD011 중복 로그인 처리
    if error_code == "CD011":
        payload["skipDup"] = "Y"
        resp = _session.post(_LOGIN_URL, data=payload, headers=headers, timeout=15)
        data = resp.json()
        error_code = data.get("_error_code", "")

    if error_code == 'CD001':
        def _session_post_read(self, **params):
            return _session.post(self.url, headers=self.headers, data=params)

        def _session_get_read(self, **params):
            return _session.get(self.url, headers=self.headers, params=params)
        webio.Post.read = _session_post_read
        webio.Get.read = _session_get_read
    else:
        logger(f">>> KRX LOGIN FAILED: {error_code}")
    return

In [ ]:
login_krx('jhlee0319', 'dwg!4r4r6y1q')

class Ticker(FnGuide):

    def __init__(self, ticker:str):
        super().__init__(ticker)
        self.date = TradingDate()
        self.freq = 'd'
        self.period = 10 # UNIT: YEARS

        # super().__init__(ohlcv)
        # if not self.is_bundle:
        #     self.viewer = TickerView(ohlcv)
        # self.viewer.ohlcv = self.data
        return

    @property
    def ohlcv(self) -> DataFrame:
        kwargs = dict(
            fromdate=self.date - 365 * self.period,
            todate=self.date.latest,
            ticker=self.ticker,
            freq=self.freq
        )
        _key = '_'.join(kwargs.values())
        if not hasattr(self, _key):
            self.__setattr__(_key, krx.get_ohlcv(**kwargs))
        return self.__getattribute__(_key)

    @property
    def annual_market_cap(self) -> Series:
        cap = self.quarterly_market_cap
        cap = cap[cap.index.str.endswith('4Q') | (cap.index == cap.index[-1])]
        cap.index = [i.replace('4Q', '12') for i in cap.index]
        cap.index.name = "year"
        return cap

    @property
    def quarterly_market_cap(self) -> Series:
        kwargs = dict(
            fromdate=self.date - 365 * self.period,
            todate=self.date.latest,
            ticker=self.ticker,
            freq='m'
        )
        _key = '_'.join(kwargs.values())
        if not hasattr(self, _key):
            market_cap = krx.get_market_cap(**kwargs)
            market_cap = market_cap[
                market_cap.index.astype(str).str.contains('03') | \
                market_cap.index.astype(str).str.contains('06') | \
                market_cap.index.astype(str).str.contains('09') | \
                market_cap.index.astype(str).str.contains('12') | \
                (market_cap.index == market_cap.index[-1])
            ]
            market_cap.index = market_cap.index.strftime("%Y/%m")
            market_cap.index = [
                col \
                    .replace("03", "1Q") \
                    .replace("06", "2Q") \
                    .replace("09", "3Q") \
                    .replace("12", "4Q") for col in market_cap.index
            ]
            market_cap.index.name = "quarter"
            self.__setattr__(_key, Series(index=market_cap.index, data=market_cap['시가총액'] / 1e+8, dtype=int))
        return self.__getattribute__(_key)


ticker = Ticker('005930')
# ticker.ohlcv
# ticker.annual_market_cap
# ticker.quarterly_market_cap